# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Exploration with `mlcroissant`
This notebook provides a practical guide for loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is described using a [Croissant](https://mlcommons.org/croissant/) schema and is accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant[full] seaborn --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata information
print("Name:", metadata.name)
print("Identifier:", getattr(metadata, 'identifier', 'N/A'))
print("Description:", metadata.description)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Date Published:", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use `@id` fields for clarity and reproducibility.

In [ ]:
# List all record sets and their fields using their @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    # If not directly present, list by traversing dataset.record_sets
    # The mlcroissant loader may populate `metadata.record_sets` instead of `metadata.recordSet`
    record_sets = getattr(dataset.metadata, 'record_sets', [])

if isinstance(record_sets, dict):
    record_sets = [record_sets]

if not record_sets:
    # Final fallback, check `dataset.record_sets` property
    record_sets = dataset.record_sets

overview = []
record_set_ids = []
for record_set in record_sets:
    # Get the @id if present
    rs_id = getattr(record_set, '@id', None) or getattr(record_set, 'id', None) or getattr(record_set, 'name', None)
    record_set_ids.append(rs_id)
    print(f'--- Record Set: {rs_id} ---')
    # Get fields (column definitions)
    fields = []
    # fields may be under record_set.field, .fields, or can be fetched via dataset.fields(record_set=...)
    if hasattr(record_set, 'field'):
        rs_fields = record_set.field
    elif hasattr(record_set, 'fields'):
        rs_fields = record_set.fields
    else:
        try:
            rs_fields = dataset.fields(record_set=rs_id)
        except Exception:
            rs_fields = []

    # Normalize fields
    if isinstance(rs_fields, dict):
        rs_fields = [rs_fields]
    for f in rs_fields or []:
        # Try to get @id and name
        f_id = getattr(f, '@id', getattr(f, 'id', getattr(f, 'name', None)))
        f_name = getattr(f, 'name', f_id)
        print(f'  Field: {f_id} (name: {f_name})')
        fields.append({'@id': f_id, 'name': f_name})
    overview.append({'@id': rs_id, 'fields': fields})
if not record_set_ids:
    print('No record sets found!')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set `@id` and field `@id`s identified above.

In [ ]:
# Extract all available record sets as DataFrames for convenience
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Preview the first DataFrame (main survey)
if dataframes:
    main_rs_id = record_set_ids[0]
    print(f"Columns in main record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data wrangling, such as filtering records, normalizing numeric fields, and grouping. All fields are referenced by their `@id`.

**Let's inspect the PHQ-9 total score field, typically used in the dataset as a numeric measurement. We'll filter out records with high scores and analyze the results.**

In [ ]:
# Identify a suitable numeric field by inspecting columns
if dataframes:
    df = dataframes[main_rs_id]
    # Try standard field names for PHQ-9 or GAD-7
    phq_fields = [col for col in df.columns if ('phq' in col.lower() and (col.lower().endswith('score') or 'total' in col.lower()))]
    gad_fields = [col for col in df.columns if ('gad' in col.lower() and (col.lower().endswith('score') or 'total' in col.lower()))]
    psq_fields = [col for col in df.columns if ('psq' in col.lower() and (col.lower().endswith('score') or 'total' in col.lower()))]
    if phq_fields:
        numeric_field_id = phq_fields[0]
        print(f"Selected numeric field (PHQ): {numeric_field_id}")
    elif gad_fields:
        numeric_field_id = gad_fields[0]
        print(f"Selected numeric field (GAD): {numeric_field_id}")
    elif psq_fields:
        numeric_field_id = psq_fields[0]
        print(f"Selected numeric field (PSQ): {numeric_field_id}")
    else:
        # Fallback: pick the first numeric column
        num_candidates = df.select_dtypes(include=np.number).columns
        if len(num_candidates)>0:
            numeric_field_id = num_candidates[0]
            print(f"Selected numeric field (automatic): {numeric_field_id}")
        else:
            raise ValueError("No suitable numeric fields found.")

    # Choose a reasonable threshold, e.g., for PHQ-9, anything above 10 is considered moderate-to-severe
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize field (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic attribute if available, e.g., 'gender', 'age_group', etc.
    candidate_groups = [col for col in df.columns if any(x in col.lower() for x in ['gender','sex','marital','village','education','age_group'])]
    if candidate_groups:
        group_field = candidate_groups[0]
        print(f"Grouped data by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].agg(['mean','count']).sort_values(by='mean',ascending=False)
        display(grouped_df.head())
    else:
        group_field = None
        print('No obvious demographic grouping field found.')

## 5. Visualization
Visualize data distributions or relationships between fields. For example, plot the distribution of the PHQ-9 (or GAD-7) scores and compare by a demographic attribute (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='royalblue', edgecolor='black')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group field exists, plot boxplot
    if group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(y=numeric_field_id, x=group_field, data=df, palette="Set2")
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated end-to-end access and analysis of the FAIR² Mental Health Survey dataset via the `mlcroissant` library. We:
- Loaded the dataset schema and data programmatically using the Croissant standard
- Explored record set structure and field definitions via their unique `@id`s
- Performed basic exploratory data analysis of mental health indicator fields (e.g., PHQ-9)
- Visualized distributions and demographic groupings where available

This approach can be extended to more advanced analyses or integrated into AI-ready workflows. Explore further by referencing `mlcroissant` [documentation](https://mlcommons.org/croissant/spec/) and dataset documentation for richer insights.
